# IT135IU - INTRODUCTION TO DATA SCIENCE
## PROJECT DELIVERABLE 3: Analysis Notebook
### Instructor: Nguyen Thi Thuy Loan
**Topic 14:** Amazon Customer Reviews Analysis (Book Category)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TeaFrost/CustomerReviewAndGrading/blob/D3_Analyze/D3_Topic14_Tra_Hao_Phu_Loi_Quy_Khoa.ipynb)

---
### TEAM MEMBERS & ROLES
1. **Nguyễn Đỗ Nam Trà** (ITDSIU25049) - Product Owner (Leader)
2. **Bùi Văn Lợi** (ITDSIU25021) - Data Engineer
3. **Nguyễn Thế Hào** (ITDSIU22139) - Data Analyst
4. **Phạm Ngọc Minh Khoa** (ITDSIU25015) - Reproducibility Lead
5. **Nguyễn Trọng Quý** (ITDSIU25037) - Visualization Lead
6. **Trần Như Phú** (ITDSIU25027) - Ethics & Privacy Officer

---
### DECISION QUESTION
> *Should Amazon product category managers invest in NLP sentiment extraction to identify product flaw patterns early, or rely solely on average star ratings?*

### NOTEBOOK OUTLINE
1. Objective, scope and hypotheses
2. Setup and data loading
3. Descriptive statistics (with Figures 1-2)
4. Statistical relationships: Part 1 Welch's t-test (Figure 3), Part 2 Spearman correlation, Part 3 linear regression (Figure 4)
5. Qualitative check: reading 10 flagged reviews
6. Interpretation
7. What this analysis does not establish
8. Conclusion and answer to the decision question

## 1. Objective, Scope and Hypotheses

**Scope.** This notebook analyses **only reviews in the `Books` category** (47,179 reviews after the D2 cleaning step). Reviews of all other products are removed before any test is run, and no comparison with other categories is made.

**Data.** `2.1_amazon_reviews_webDisplay.csv`, the cleaned dataset produced in Deliverable 2. D2 kept only verified purchases with at least 3 helpful votes, and added the column `high_rating_negative_flag` (1 if a 4-5 star review contains a negative keyword such as "bad" or "hate", otherwise 0).

**Why this matters for the decision question.** If star ratings already tell us everything, NLP is not needed. If reviews with the same star rating differ in what they say, then text analysis may add information. The three relationships below test this from different angles, using `scipy`:

| Part | Method | Question (Books reviews only) | H0 |
|---|---|---|---|
| 1 | Welch's t-test | Among book reviews rated 4 or 5 stars, is the average star rating of reviews that contain a negative word different from the average of reviews that do not? | Both groups have the same mean star rating |
| 2 | Spearman correlation | Is the number of helpful votes on a book review related to its length? | No monotonic relationship (rho = 0) |
| 3 | Simple linear regression | How well does review length predict the number of helpful votes? | Slope = 0 |

**A note on p-values.** With 47,179 book reviews, even very small differences are "statistically significant". For every result we therefore also report an effect size, which measures how strong the relationship is. When Python prints `p = 0`, the value is too small to represent, and we report it as p < 0.001.

## 2. Setup and Data Loading

In [ ]:
# --- Setup -----------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
np.random.seed(SEED)
ALPHA = 0.05

plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = "2.1_amazon_reviews_webDisplay.csv"
FILE_ID = "1Y_aYSFOty8bo7j-2yoeNTMTFPUM-dt4y"   # public Google Drive file

# Download the dataset only if it is not already in the working folder
if not os.path.exists(DATA_PATH):
    print("Downloading dataset from public Google Drive ...")
    try:
        import gdown
    except ImportError:
        %pip install -q gdown
        import gdown
    gdown.download(id=FILE_ID, output=DATA_PATH, quiet=False)
    print("Download completed.")

# --- Load and keep the Books category only ---------------------------------
rel_all = pd.read_csv(DATA_PATH)
rel = rel_all[rel_all["product_category"] == "Books"].copy()
assert len(rel) > 0, "No 'Books' rows found - check the spelling of the category name."

# Fail loudly if a column we rely on is missing (instead of silently drawing something else)
REQUIRED = ["star_rating", "review_length", "helpful_votes",
            "high_rating_negative_flag", "review_body"]
missing = [c for c in REQUIRED if c not in rel.columns]
assert not missing, f"Missing columns: {missing}"

def fmt_p(p):
    """Python prints p = 0 when the value underflows, so report it as p < 0.001."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

print(f"Books reviews: {len(rel):,} rows, {rel.shape[1]} columns")
print(f"pandas {pd.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed = {SEED}")

## 3. Descriptive Statistics

Before testing anything, we describe the data in terms of the decision question: how are star ratings distributed, how many high-rated reviews contain a negative word, and how do the two groups of high-rated reviews look side by side?

### 3.1 Star ratings and negative-word flag

In [ ]:
# --- 3.1 Star rating distribution (Books) ----------------------------------
star_counts = rel["star_rating"].value_counts().sort_index()
star_table = pd.DataFrame({
    "reviews": star_counts,
    "share_%": (100 * star_counts / len(rel)).round(1),
})
print("Star rating distribution (all Books reviews)")
print(star_table)

# --- High-rated reviews (4-5 stars) with and without a negative word -------
high = rel[rel["star_rating"] >= 4].copy()
n_high = len(high)
n_flag = int((high["high_rating_negative_flag"] == 1).sum())
print(f"\nHigh-rated reviews (4-5 stars): {n_high:,}")
print(f"  containing a negative word  : {n_flag:,} ({100 * n_flag / n_high:.1f}%)")

group_summary = (
    high.groupby("high_rating_negative_flag")
        .agg(reviews=("star_rating", "size"),
             mean_rating=("star_rating", "mean"),
             pct_5_star=("star_rating", lambda s: 100 * (s == 5).mean()),
             median_length=("review_length", "median"),
             median_helpful_votes=("helpful_votes", "median"))
        .round(2)
)
group_summary.index = group_summary.index.map({0: "No negative word", 1: "Negative word"})
print("\nHigh-rated reviews: side-by-side comparison")
print(group_summary)

In [ ]:
# --- Figure 1: star rating distribution ------------------------------------
shares = 100 * star_counts / len(rel)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(star_counts.index.astype(int).astype(str), shares.values, color="#4C78A8")
for bar, v in zip(bars, shares.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, shares.max() * 1.15)          # bars start at zero
ax.set_title(f"Figure 1. Star rating distribution of Books reviews (n = {len(rel):,})",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Star rating")
ax.set_ylabel("Share of reviews (%)")
plt.tight_layout()
plt.show()

### 3.2 Review length and helpful votes

In [ ]:
# --- 3.2 Summary statistics of the two numeric variables used later ---------
print(rel[["review_length", "helpful_votes"]].describe().round(1))

In [ ]:
# --- Figure 2: distribution of review length (log-spaced bins, nothing trimmed)
assert rel["review_length"].min() > 0, "Log axis needs review_length > 0"
bins = np.logspace(np.log10(rel["review_length"].min()),
                   np.log10(rel["review_length"].max()), 50)
median_len = rel["review_length"].median()

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(rel["review_length"], bins=bins, color="#F58518", edgecolor="white")
ax.set_xscale("log")
ax.axvline(median_len, color="black", linestyle="--", linewidth=1,
           label=f"Median = {median_len:,.0f} characters")
ax.set_title("Figure 2. Distribution of review length, Books (all reviews, log scale)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Review length (characters, log scale)")
ax.set_ylabel("Number of reviews")
ax.legend()
plt.tight_layout()
plt.show()

**What the descriptive statistics show.**
- The analysis covers **47,179** book reviews. Of the **27,755** reviews rated 4-5 stars, **3,818 (13.8%)** contain a negative keyword, so roughly one in seven high-rated reviews says something that a star average cannot reveal.
- Both numeric variables are strongly right-skewed. The median review has 646 characters but the longest has 43,840, and the median review has 5 helpful votes but the maximum is 27,550. This is why we use rank-based and log-scale methods in Parts 2 and 3, and why Figure 2 uses a log axis instead of cutting off the long tail.
- Figure 1 and the group table above show that the flagged and unflagged high-rated reviews have very similar star ratings. Part 1 tests whether the small gap is real.

## 4. Statistical Relationships

### Part 1. Welch's t-test: star rating of high-rated book reviews with vs without a negative word

In [ ]:
# --- Part 1. T-test ----------------------------------------------------------
# Group A: 4-5 star book reviews whose text contains a negative keyword (D2 flag = 1)
# Group B: 4-5 star book reviews whose text does not (D2 flag = 0)
# H0: both groups have the same mean star rating.
a = high.loc[high["high_rating_negative_flag"] == 1, "star_rating"]
b = high.loc[high["high_rating_negative_flag"] == 0, "star_rating"]

# Welch's t-test: compares two means without assuming equal variances.
t, p = stats.ttest_ind(a, b, equal_var=False)

# Cohen's d: difference in means in standard-deviation units (0.2 small, 0.5 medium, 0.8 large).
pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
d = (a.mean() - b.mean()) / pooled_sd
diff = a.mean() - b.mean()

print(f"with negative word   : n = {len(a):,}, mean rating = {a.mean():.3f}")
print(f"without negative word: n = {len(b):,}, mean rating = {b.mean():.3f}")
print(f"difference = {diff:.3f} stars, t = {t:.1f}, {fmt_p(p)}, Cohen's d = {d:.3f}")
print("Reject H0" if p < ALPHA else "Fail to reject H0")

In [ ]:
# --- Figure 3: share of 5-star reviews, with vs without a negative word ------
# Uses exactly the same two groups as the t-test above.
labels = ["No negative word", "Negative word"]
pct_5 = [100 * (b == 5).mean(), 100 * (a == 5).mean()]
ns = [len(b), len(a)]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
bars = ax.bar(labels, pct_5, color=["#4C78A8", "#E45756"])
for bar, v, n in zip(bars, pct_5, ns):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v:.1f}%\n(n = {n:,})",
            ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 100)                            # full 0-100% axis: no exaggerated gap
ax.set_title("Figure 3. Share of 5-star reviews among 4-5 star Books reviews",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Does the review text contain a negative word?")
ax.set_ylabel("Share rated 5 stars (%)")
plt.tight_layout()
plt.show()

**Part 1 result.** Among 4-5 star book reviews, the 3,818 reviews containing a negative keyword have a mean rating of 4.749, compared with 4.826 for the 23,937 reviews that do not. The difference is 0.078 stars, and the t-test rejects the null hypothesis that the two means are equal (t = -10.4, p < 0.001).

Because every review in this group has either 4 or 5 stars, the mean is easy to read: a mean of 4.749 means that about 75% of the flagged reviews give 5 stars, compared with about 83% of the reviews without a negative keyword (Figure 3). Readers who use a negative word are therefore slightly more likely to give 4 stars instead of 5.

The difference is real but small: Cohen's d is -0.20, exactly at the conventional threshold for a small effect. In practical terms, a high-rated book review that contains a negative word receives almost the same star rating as one that does not. The star rating cannot separate the two groups, so whatever the negative word expresses is visible only in the review text.

### Part 2. Spearman correlation: helpful votes vs review length

In [ ]:
# --- Part 2. Correlation: helpful votes vs review length (Books) ------------
# helpful_votes is highly skewed (see section 3.2), so we use Spearman's correlation,
# which is rank-based and robust to a few extreme reviews.
rho, p_rho = stats.spearmanr(rel["review_length"], rel["helpful_votes"])
print(f"Spearman correlation = {rho:.3f}, {fmt_p(p_rho)}")

**Part 2 result.** The Spearman correlation between review length and helpful votes for book reviews is 0.267 (p < 0.001). The correlation is positive, meaning longer book reviews tend to receive more helpful votes, but it is weak (below 0.3). Many short reviews still receive many votes and many long reviews receive few, so review length on its own tells us only a little about how helpful readers find a book review.

The summary table in section 3.2 shows why we use Spearman's rank correlation: helpful votes have a median of 5 but a maximum of 27,550, so a few extreme reviews would dominate an ordinary (Pearson) correlation.

### Part 3. Simple linear regression: review length -> helpful votes

In [ ]:
# --- Part 3. Simple linear regression: review length -> helpful votes (Books) -
# Both variables are right-skewed, so we fit the line on their logarithms:
#     log(helpful_votes) = intercept + slope * log(review_length)
# (D2 kept only reviews with >= 3 helpful votes; rows with length 0 are dropped so the log is defined.)
reg = rel[(rel["review_length"] > 0) & (rel["helpful_votes"] > 0)]
fit = stats.linregress(np.log(reg["review_length"]), np.log(reg["helpful_votes"]))
r2 = fit.rvalue ** 2
pct = 100 * (2 ** fit.slope - 1)

print(f"n = {len(reg):,}")
print(f"slope = {fit.slope:.3f}, intercept = {fit.intercept:.3f}")
print(f"R squared = {r2:.3f}, {fmt_p(fit.pvalue)}")
print(f"a review twice as long gets about {abs(pct):.1f}% {'more' if pct >= 0 else 'fewer'} helpful votes on average")

In [ ]:
# --- Figure 4: review length vs helpful votes, with the fitted line ----------
# The scatter shows a random sample of 2,000 reviews (readability); the line is fitted on ALL reviews in `reg`.
sample_df = reg.sample(min(2000, len(reg)), random_state=SEED)
x_line = np.logspace(np.log10(reg["review_length"].min()), np.log10(reg["review_length"].max()), 100)
y_line = np.exp(fit.intercept) * x_line ** fit.slope

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(sample_df["review_length"], sample_df["helpful_votes"],
           alpha=0.35, s=14, color="teal", label="Random sample of 2,000 reviews")
ax.plot(x_line, y_line, color="crimson", linewidth=2,
        label=f"Fitted line (slope = {fit.slope:.2f}, R² = {r2:.3f}, fitted on all {len(reg):,} reviews)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Figure 4. Review length vs helpful votes, Books (both axes log scale)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Review length (characters, log scale)")
ax.set_ylabel("Helpful votes (log scale)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Part 3 result.** The fitted line has a slope of 0.269 on the log scale. Because both variables are on a log scale, the slope can be read as a percentage: a book review twice as long receives about 20.5% more helpful votes on average (p < 0.001).

R² is 0.088, so review length explains only about 8.8% of the variation in helpful votes on book reviews. The other 91% depends on factors that are not in this model. Figure 4 shows the same picture: the line slopes upward, but the points are spread widely around it. The regression confirms the relationship found in Part 2 and shows how weak it is: longer book reviews do get somewhat more helpful votes, but review length is a poor predictor of how helpful a particular book review will be.

## 5. Qualitative Check: Reading 10 Flagged Reviews

The t-test only tells us that flagged reviews differ slightly in star rating. It does not tell us whether the negative word reflects a real complaint. To check this, we read 10 randomly selected flagged reviews in full (the row index on the left is the review ID).

In [ ]:
# --- Random sample of 10 flagged book reviews, full text ---------------------
pd.set_option("display.max_colwidth", None)          # no character limit
sample = high[high["high_rating_negative_flag"] == 1].sample(10, random_state=SEED)

full = sample[["star_rating", "review_body"]].copy()
full["review_body"] = full["review_body"].str.replace("<br />", "\n", regex=False)

(full.style
     .set_properties(subset=["review_body"], **{"white-space": "pre-wrap", "text-align": "left", "min-width": "700px"})
     .set_properties(subset=["star_rating"], **{"text-align": "center"})
     .format({"star_rating": "{:.0f}"}))

**What the sample shows.** We judged whether the negative wording reflects a genuine complaint about the book (for example its content, writing, editing, printing or price), or whether it appears for another reason.

| Review ID | Stars | Negative wording in context | Genuine complaint? |
|---|---|---|---|
| 223734 | 4 | "bad things", "bad genes", "worst offenders"; also "This is not a perfect book", "skim the first 400 or so pages" | Yes, partly: the keywords describe the book's topic, but the review criticises the book's structure and argument |
| 273587 | 5 | "really good (bad) humor"; "this book is basic" | No, playful praise; "basic" is advice on who the book suits |
| 259161 | 5 | "Don't be boring", "The worst presentations" | No, summarises and quotes the book's advice |
| 307133 | 5 | "Rich Dad, Poor Dad", "the middle class and poor" | No, the title of another book and the book's topic |
| 291122 | 5 | "a couple of bad reviews here" | No, disagrees with other reviewers |
| 201837 | 5 | "getting a bad rap", "I hated Kakutani's review" | No, defends the book against a critic |
| 292593 | 5 | "completely wrong about him", "worse-off", "annoying" | No, about the reviewer's first impression and a different author |
| 296297 | 5 | "exaggerated, made up", "ripping each other to shreds" | No, dismisses the controversy and defends the book |
| 271160 | 5 | "eating meat is cruel", "true horror and evil" | No, describes the book's subject (the food industry) |
| 279719 | 5 | "never a dull word", "a poor farm kid", "bad guys" | No, negated, and describes the story |

**Count: 1 of 10 is a genuine complaint; 9 are not.** The only review that really criticises the book is also the only 4-star review in the sample, which is consistent with Part 1 (flagged reviews give 4 stars slightly more often). Ten reviews are far too few to draw a conclusion, but the pattern is clear: in book reviews, negative words are mostly used to describe the book's subject, to disagree with other reviewers or critics, or in a negated or playful way. A flag should therefore be read as "contains a negative word", not as "reports a problem with the book".

## 6. Interpretation

Two findings are relevant to the decision question for **Books**.

First, about one in seven high-rated book reviews (3,818 of 27,755, or 13.8%) contains a negative keyword, yet their star rating is almost identical to that of the other reviews (4.75 vs 4.83, a small effect with Cohen's d = -0.20). A dashboard based on average star ratings cannot tell these book reviews apart.

Second, what readers find helpful in a book review is only weakly related to a simple measure such as review length (Spearman correlation 0.27, R² = 0.088). Doubling the length of a review adds about 20% more helpful votes on average, but length explains less than a tenth of the variation, so the useful content of a book review cannot be summarised by a single number either.

Both findings support analysing book review text alongside star ratings. They do not yet show that the text reveals problems with the books: in the sample we read, only 1 of 10 flagged reviews contained a genuine complaint about the book.

## 7. What This Analysis Does Not Establish

- **These are associations, not causal effects.** We found patterns, not causes. We have not shown that negative wording lowers a book's star rating, or that writing a longer review earns more helpful votes.
- **A flag does not always mean a complaint about the book.** We read 10 randomly selected flagged book reviews (section 5), and only 1 was a genuine complaint. The other 9 were flagged because the review described the book's subject, disagreed with other reviewers or a critic, criticised a different author, or used a negative word in a negated or playful way. So the real number of complaints is much lower than 3,818, but 10 reviews are too few to say how much lower.
- **Statistical significance here mainly reflects sample size.** With 47,179 book reviews, even tiny differences are significant. What matters is the effect size, and all three effect sizes are small or weak.
- **The t-test is a rough tool in this setting.** Within the 4-5 star group, the star rating takes only two values, so the data are far from normally distributed. With samples this large the test remains valid, but it essentially restates the difference between two proportions (about 75% vs 83% five-star reviews).
- **The sample is filtered.** D2 kept only verified purchases with at least 3 helpful votes. Book reviews with 0 to 2 votes are excluded, so the true relationship between review length and helpfulness across all book reviews may differ from what we measured.
- **Some review texts contain duplicated passages.** Two of the ten reviews we read (273587 and 223734) repeat whole paragraphs, which looks like a problem in the source data. This inflates `review_length` for such reviews and adds noise to Parts 2 and 3.
- **The results apply to Books only.** They should not be read as describing any other product category.
- **It says nothing about returns or sales.** The dataset contains no return or revenue variable, so we cannot test whether acting on book review text would reduce return rates.

## 8. Conclusion and Answer to the Decision Question

> *Should Amazon product category managers invest in NLP sentiment extraction to identify product flaw patterns early, or rely solely on average star ratings?*

* **Finding 1 (Welch's t-test).** Among 4-5 star book reviews, those containing a negative word have a slightly lower mean rating (4.75 vs 4.83; p < 0.001, Cohen's d = -0.20, a small effect). Star ratings therefore barely distinguish the 13.8% of high-rated reviews that contain negative wording from the rest.
* **Finding 2 (Spearman correlation and linear regression).** Review length has a weak positive association with helpful votes (rho = 0.27, R² = 0.088). Length alone is a poor predictor of how helpful a review is.
* **Finding 3 (qualitative check).** A simple negative-keyword flag is not a reliable complaint detector: only 1 of 10 flagged reviews was a genuine complaint about the book.

**Answer.** For the **Books** category, average star ratings alone cannot show what 4-5 star reviews say in their text, so relying *solely* on them leaves that information unused. However, this analysis does **not** show that the text contains early warning signs of product flaws, and our keyword flag is too noisy to be used as a flaw detector. We therefore recommend a **small, validated pilot** rather than a full investment: use a better method than keyword matching (for example sentiment or aspect-based classification), check its precision on a larger, manually labelled sample of book reviews, and only then decide whether the benefit justifies the cost. Whether acting on review text would reduce returns or improve sales cannot be answered with this dataset.